In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_key", "")
dbutils.widgets.text("load_type", "")

merge_key = dbutils.widgets.get("merge_key")
load_type = dbutils.widgets.get("load_type")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")

print("=" * 80)
print("STEP 1: Reading job parameters")
print("=" * 80)
print(f"  table_name        : '{table_name}'")
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  load_type         : '{load_type}'")
print(f"  merge_key         : '{merge_key}'")
if not table_name:
    print("  WARNING: table_name is empty")
if not landing_timestamp:
    print("  WARNING: landing_timestamp is empty")
if not load_type:
    print("  WARNING: load_type is empty (expected FULL, APPEND or MERGE)")

In [0]:
source_table = f'adb_ws_wecare.bronze.{table_name}'
filter_condition = f'landing_timestamp = "{landing_timestamp}"'

print("=" * 80)
print("STEP 2: Reading data from bronze layer")
print("=" * 80)
print(f"  Source table     : {source_table}")
print(f"  Filter condition : {filter_condition}")
print("  Equivalent SQL query:")
print(f"    SELECT *\n    FROM {source_table}\n    WHERE {filter_condition}")

bronze_df=spark.read.table(source_table) \
    .filter(filter_condition)

bronze_count = bronze_df.count()
print(f"  Columns ({len(bronze_df.columns)}): {bronze_df.columns}")
print(f"  Records read from bronze: {bronze_count}")
if bronze_count == 0:
    print("  WARNING: No records found in bronze for the given landing_timestamp")

In [0]:
from pyspark.sql.functions import *;
updated_df=bronze_df.withColumn('insert_timestamp', current_timestamp()) \
    .withColumn('update_timestamp', current_timestamp())

print("=" * 80)
print("STEP 3: Adding audit columns")
print("=" * 80)
print("  Added columns: insert_timestamp = current_timestamp(), update_timestamp = current_timestamp()")
print("  Equivalent SQL query:")
print(f"    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp\n    FROM {source_table}\n    WHERE {filter_condition}")
print(f"  Final columns ({len(updated_df.columns)}): {updated_df.columns}")

In [0]:
from delta.tables import DeltaTable

target_table = f'adb_ws_wecare.silver.{table_name}'
load_type_norm = load_type.strip().upper()
target_exists = spark.catalog.tableExists(target_table)

print("=" * 80)
print("STEP 4: Writing data to silver layer")
print("=" * 80)
print(f"  Target table        : {target_table}")
print(f"  Load type (normal.) : {load_type_norm}")
print(f"  Target table exists : {target_exists}")
col_list = ', '.join([f'`{c}`' for c in updated_df.columns])

if load_type_norm == 'FULL':
    # Full load: overwrite the target table
    print("  Mode: FULL -> overwriting the target table")
    print("  Equivalent SQL query:")
    print(f"    CREATE OR REPLACE TABLE {target_table} AS\n    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp\n    FROM {source_table}\n    WHERE {filter_condition}")
    updated_df.write.mode('overwrite').saveAsTable(target_table)
    print(f"  FULL load completed for {target_table}")

elif load_type_norm == 'APPEND':
    # Append: add the new records to the target table (creates it if missing)
    print("  Mode: APPEND -> adding new records to the target table" + ("" if target_exists else " (table will be created)"))
    print("  Equivalent SQL query:")
    print(f"    INSERT INTO {target_table} ({col_list})\n    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp\n    FROM {source_table}\n    WHERE {filter_condition}")
    updated_df.write.mode('append').saveAsTable(target_table)
    print(f"  APPEND load completed for {target_table}")

elif load_type_norm == 'MERGE':
    # Merge: update existing records and insert new ones based on merge_key
    print("  Mode: MERGE -> upserting records based on merge_key")
    merge_keys = [k.strip() for k in merge_key.split(',') if k.strip()]
    print(f"  Parsed merge keys: {merge_keys}")
    if not merge_keys:
        raise ValueError("merge_key must be provided when load_type is MERGE")
    missing = [k for k in merge_keys if k not in updated_df.columns]
    if missing:
        raise ValueError(f"merge_key column(s) not found in source: {missing}")

    # Remove duplicate keys in the source batch (MERGE fails if multiple source rows match one target row)
    source_df = updated_df.dropDuplicates(merge_keys)
    src_count = updated_df.count()
    dedup_count = source_df.count()
    print(f"  Source records before de-duplication : {src_count}")
    print(f"  Source records after de-duplication  : {dedup_count}")
    print(f"  Duplicate records removed            : {src_count - dedup_count}")

    if not target_exists:
        print(f"  Target table does not exist -> creating {target_table} with source data")
        print("  Equivalent SQL query:")
        print(f"    CREATE OR REPLACE TABLE {target_table} AS\n    SELECT * FROM <deduplicated source on {merge_keys}>")
        source_df.write.mode('overwrite').saveAsTable(target_table)
        print(f"  Initial load completed for {target_table}")
    else:
        merge_condition = ' AND '.join([f't.`{k}` = s.`{k}`' for k in merge_keys])
        # On update keep the original insert_timestamp; refresh all other columns
        update_set = {f'`{c}`': f's.`{c}`' for c in source_df.columns if c != 'insert_timestamp'}
        set_clause = ',\n        '.join([f't.{k} = {v}' for k, v in update_set.items()])
        print(f"  Merge condition: {merge_condition}")
        print("  Equivalent SQL query:")
        print(f"    MERGE INTO {target_table} AS t\n    USING <deduplicated source on {merge_keys}> AS s\n    ON {merge_condition}\n    WHEN MATCHED THEN UPDATE SET\n        {set_clause}\n    WHEN NOT MATCHED THEN INSERT *")
        (DeltaTable.forName(spark, target_table).alias('t')
            .merge(source_df.alias('s'), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsertAll()
            .execute())
        print(f"  MERGE completed for {target_table}")
        # Show operation metrics from the latest Delta history entry
        last_op = DeltaTable.forName(spark, target_table).history(1).select('operation', 'operationMetrics').collect()[0]
        metrics = last_op['operationMetrics'] or {}
        print(f"  Operation: {last_op['operation']}")
        print(f"    Rows inserted : {metrics.get('numTargetRowsInserted', 'n/a')}")
        print(f"    Rows updated  : {metrics.get('numTargetRowsUpdated', 'n/a')}")
        print(f"    Rows deleted  : {metrics.get('numTargetRowsDeleted', 'n/a')}")
    updated_df = source_df

else:
    print(f"  ERROR: Unsupported load_type '{load_type}'")
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected FULL, APPEND or MERGE")

In [0]:
record_count = updated_df.count()
print("=" * 80)
print("STEP 5: Finishing notebook")
print("=" * 80)
print(f"  Records processed into {target_table}: {record_count}")
print(f"  Exiting notebook with value: {record_count}")
dbutils.notebook.exit(str(record_count))